# Gerar dataset sintético com 5.000 cenas usando todos os silos

In [1]:
from pathlib import Path
import shutil
import cv2
import numpy as np
import yaml
import tifffile

CONFIGS_DIR = Path("../configs")
OUT_DIR = Path("../out")

SURFACES_YAML = CONFIGS_DIR / "catalogo_superficies.yaml"
SENSOR_YAML = CONFIGS_DIR / "config_sensor.yaml"
SILOS_YAML = CONFIGS_DIR / "config_silos.yaml"

DATASET_ROOT = OUT_DIR / "synthetic_depth_dataset"
CONFIG_DIR = DATASET_ROOT / "config"
SCENES_DIR = DATASET_ROOT / "scenes"

SEED = 42
DATASET_SIZE = 5000

GRID_SIZE = 160
RAY_MARCH_STEPS = 500
ROOF_CLEARANCE_M = 0.05

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SCENES_DIR.mkdir(parents=True, exist_ok=True)

#falta documentacao completa do dataset - exemplo: o tipo de arquivo - .tiif em 16 bits 


In [3]:
with SURFACES_YAML.open("r", encoding="utf-8") as f:
    superficies = yaml.safe_load(f)["superficies"]
with SENSOR_YAML.open("r", encoding="utf-8") as f:
    sensor_config = yaml.safe_load(f)
with SILOS_YAML.open("r", encoding="utf-8") as f:
    silos = yaml.safe_load(f)["silos"]

sensor = sensor_config["sensor"]
pose_ranges = sensor_config["pose_ranges"]

print(f"Superfícies: {len(superficies)}")
print(f"Silos: {len(silos)}")
print([s["model"] for s in silos])


Superfícies: 45
Silos: 6
['36-02-60', '30-04-60', '30-4,5-60', '36-03-60', '36-3,5-60', 'ovos_mombuca']


In [4]:
def preparar_dominio(radius_m, grid_size):
    eixo = np.linspace(-radius_m, radius_m, grid_size)
    X, Z = np.meshgrid(eixo, eixo)
    RAIO = np.sqrt(X**2 + Z**2)
    MASCARA_SILO = RAIO <= radius_m
    return eixo, X, Z, RAIO, MASCARA_SILO

def rugosidade(X, Z, lambda_m, beta_deg):
    beta = np.deg2rad(beta_deg)
    ub = X * np.cos(beta) + Z * np.sin(beta)
    vb = -X * np.sin(beta) + Z * np.cos(beta)
    return 0.62 * np.sin(2*np.pi*ub/lambda_m) + 0.38 * np.cos(4*np.pi*vb/lambda_m)

def base_automatica(relevo, mascara_silo, H):
    valores = relevo[mascara_silo]
    return H/2 - (np.nanmin(valores) + np.nanmax(valores))/2

def finalizar_superficie(relevo, parametros, X, Z, mascara_silo, H):
    nu = parametros["nu"]
    b = base_automatica(relevo, mascara_silo, H) if nu == "auto" else nu * H
    Y = b + relevo + parametros["a0_m"] * rugosidade(X, Z, parametros["lambda_m"], parametros["beta_deg"])
    Y = np.clip(Y, 0, H)
    return np.where(mascara_silo, Y, np.nan)


In [5]:
def gerar_superficie(item, radius_m, cylinder_height_m, X, Z, RAIO, mascara_silo):
    tipo = item["tipo"]
    p = item["parametros"]
    r = RAIO
    R = radius_m
    H = cylinder_height_m
    A = 0.4 * R

    if tipo == "Plana (assentada)":
        return finalizar_superficie(np.zeros_like(X), p, X, Z, mascara_silo, H)

    if tipo == "Monte lateral com cratera central":
        epsilon = p["epsilon"]; alpha = p["alpha"]; depth_factor = p["depth_factor"]
        theta = np.deg2rad(p["theta_deg"]); phi = np.deg2rad(p["phi_deg"])
        cx = epsilon * R * np.cos(theta); cz = epsilon * R * np.sin(theta)
        distancia = np.sqrt((X-cx)**2 + (Z-cz)**2)
        monte = np.tan(phi) * np.maximum(0, alpha*R - distancia)
        cratera = -depth_factor * A + np.tan(phi) * r
        return finalizar_superficie(np.minimum(monte, cratera), p, X, Z, mascara_silo, H)

    if tipo == "Cratera de descarga":
        rho = p["rho"]; depth_factor = p["depth_factor"]; phi = np.deg2rad(p["phi_deg"])
        D = min(depth_factor*A, rho*R*np.tan(phi))
        F = max(0, rho*R - D/np.tan(phi))
        relevo = np.minimum(0, -D + np.tan(phi) * np.maximum(0, r-F))
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Monte deslocado (boca lateral)":
        alpha = p["alpha"]; c1, c2 = p["c"]; phi = np.deg2rad(p["phi_deg"])
        distancia = np.sqrt((X-R*c1)**2 + (Z-R*c2)**2)
        relevo = np.tan(phi) * np.maximum(0, alpha*R - distancia)
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Plano inclinado (escorregada)":
        k = p["k"]; phi = np.deg2rad(p["phi_deg"]); theta = np.deg2rad(p["theta_deg"])
        relevo = np.tan(k*phi) * (X*np.cos(theta) + Z*np.sin(theta))
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Cratera excêntrica (canal de fluxo lateral)":
        epsilon = p["epsilon"]; rho = p["rho"]; depth_factor = p["depth_factor"]
        theta = np.deg2rad(p["theta_deg"]); phi = np.deg2rad(p["phi_deg"])
        cx = epsilon * R * np.cos(theta); cz = epsilon * R * np.sin(theta)
        distancia = np.sqrt((X-cx)**2 + (Z-cz)**2)
        D = min(depth_factor*A, rho*R*np.tan(phi))
        F = max(0, rho*R - D/np.tan(phi))
        relevo = np.minimum(0, -D + np.tan(phi) * np.maximum(0, distancia-F))
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Frente de avalanche (dois patamares)":
        h = p["h"]; x0 = p["x0"]; theta = np.deg2rad(p["theta_deg"]); phi = np.deg2rad(p["phi_deg"])
        u = X*np.cos(theta) + Z*np.sin(theta) - x0*R
        kappa = h*H/(2*np.tan(phi))
        relevo = (h*H/2) * np.tanh(u/kappa)
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Cone truncado (topo achatado)":
        h_t = p["h_t"]; epsilon = p["epsilon"]; theta = np.deg2rad(p["theta_deg"]); phi = np.deg2rad(p["phi_deg"])
        cx = epsilon * R * np.cos(theta); cz = epsilon * R * np.sin(theta)
        distancia = np.sqrt((X-cx)**2 + (Z-cz)**2)
        relevo = np.minimum(h_t*R*np.tan(phi), np.tan(phi) * np.maximum(0, R-distancia))
        return finalizar_superficie(relevo, p, X, Z, mascara_silo, H)

    if tipo == "Rathole (canal central)":
        rho_0 = p["rho_0"]; d_star = p["d_star"]; n = p["n"]; nu = p["nu"]; H0 = 0.0
        b = nu*H
        Y = b - d_star*(b-H0)/(1 + (r/(rho_0*R))**n) + p["a0_m"] * rugosidade(X, Z, p["lambda_m"], p["beta_deg"])
        Y = np.clip(Y, 0, H)
        return np.where(mascara_silo, Y, np.nan)

    raise ValueError(f"Tipo não implementado: {tipo}")


In [6]:
def validar_silo(silo):
    g = silo["geometry"]
    roof_top_y = g["cylinder_height_m"] + g["roof_height_m"]
    bottom_y = -(g["hopper_height_m"] + g["outlet_height_m"])
    total = roof_top_y - bottom_y
    if not np.isclose(total, g["total_height_m"], atol=1e-4):
        raise ValueError(f"Soma geométrica inconsistente para {silo['model']}: {total:.4f} != {g['total_height_m']:.4f}")
    return roof_top_y, bottom_y

def roof_max_y(radial_distance_m, geometry):
    R = geometry["radius_m"]
    y_base = geometry["cylinder_height_m"]
    y_top = geometry["cylinder_height_m"] + geometry["roof_height_m"]
    r_top = geometry["roof_top_radius_m"]
    if radial_distance_m <= r_top:
        return y_top
    t = (radial_distance_m - r_top) / (R - r_top) if not np.isclose(R, r_top) else 0.0
    return y_top + t * (y_base - y_top)


In [7]:
def bilinear_surface_y(Y, eixo, x_m, z_m):
    x_m = float(np.clip(x_m, eixo[0], eixo[-1]))
    z_m = float(np.clip(z_m, eixo[0], eixo[-1]))
    ix = np.searchsorted(eixo, x_m)
    iz = np.searchsorted(eixo, z_m)
    ix = min(max(ix, 1), len(eixo)-1)
    iz = min(max(iz, 1), len(eixo)-1)

    x0, x1 = eixo[ix-1], eixo[ix]
    z0, z1 = eixo[iz-1], eixo[iz]
    q11 = Y[iz-1, ix-1]; q21 = Y[iz-1, ix]; q12 = Y[iz, ix-1]; q22 = Y[iz, ix]
    vals = np.array([q11, q21, q12, q22], dtype=float)

    if np.any(np.isnan(vals)):
        Xg, Zg = np.meshgrid(eixo, eixo)
        dist2 = (Xg - x_m)**2 + (Zg - z_m)**2
        valid = np.isfinite(Y)
        j, i = np.unravel_index(np.argmin(np.where(valid, dist2, np.inf)), dist2.shape)
        return float(Y[j, i])

    tx = (x_m - x0)/(x1 - x0) if x1 != x0 else 0.0
    tz = (z_m - z0)/(z1 - z0) if z1 != z0 else 0.0
    return float((1-tx)*(1-tz)*q11 + tx*(1-tz)*q21 + (1-tx)*tz*q12 + tx*tz*q22)


In [8]:
def pose_aleatoria_valida(Y, eixo, geometry, rng):
    R = geometry["radius_m"]
    for _ in range(1000):
        sensor_height_fraction = rng.uniform(pose_ranges["sensor_height_fraction"]["min"], pose_ranges["sensor_height_fraction"]["max"])
        azimuth_deg = rng.uniform(pose_ranges["azimuth_deg"]["min"], pose_ranges["azimuth_deg"]["max"])
        radial_offset_fraction = rng.uniform(pose_ranges["radial_offset_fraction"]["min"], pose_ranges["radial_offset_fraction"]["max"])
        tilt_deg = rng.uniform(pose_ranges["tilt_deg"]["min"], pose_ranges["tilt_deg"]["max"])
        radial_offset_m = radial_offset_fraction * R
        azimuth_rad = np.deg2rad(azimuth_deg)
        sensor_x = radial_offset_m * np.cos(azimuth_rad)
        sensor_z = radial_offset_m * np.sin(azimuth_rad)
        surface_y_m = bilinear_surface_y(Y, eixo, sensor_x, sensor_z)
        roof_y_m = roof_max_y(radial_offset_m, geometry)
        min_sensor_y = surface_y_m + sensor["min_range_m"]
        max_sensor_y = roof_y_m - ROOF_CLEARANCE_M
        if min_sensor_y >= max_sensor_y:
            continue
        sensor_y = min_sensor_y + sensor_height_fraction * (max_sensor_y - min_sensor_y)
        tilt_rad = np.deg2rad(tilt_deg)
        inward = np.array([-np.cos(azimuth_rad), 0.0, -np.sin(azimuth_rad)])
        downward = np.array([0.0, -1.0, 0.0])
        direction = np.cos(tilt_rad) * downward + np.sin(tilt_rad) * inward
        direction = direction / np.linalg.norm(direction)
        return {
            "sensor_height_fraction": float(sensor_height_fraction),
            "azimuth_deg": float(azimuth_deg),
            "radial_offset_fraction": float(radial_offset_fraction),
            "tilt_deg": float(tilt_deg),
            "position": np.array([sensor_x, sensor_y, sensor_z], dtype=float),
            "direction": direction.astype(float),
            "surface_y_m": float(surface_y_m),
            "roof_y_m": float(roof_y_m),
            "radial_offset_m": float(radial_offset_m),
        }
    raise RuntimeError("Não foi possível amostrar uma pose válida.")


In [9]:
def pixel_directions_32():
    n = sensor["high_size"]
    diag = np.deg2rad(sensor["fov_diagonal_deg"])
    half_hv = np.arctan(np.tan(diag / 2) / np.sqrt(2))
    coords = np.linspace(-1, 1, n)
    dirs = np.zeros((n, n, 3), dtype=float)
    for j, v in enumerate(coords):
        for i, u in enumerate(coords):
            x = np.tan(half_hv) * u
            z = np.tan(half_hv) * v
            vec = np.array([x, 1.0, z], dtype=float)
            dirs[j, i] = vec / np.linalg.norm(vec)
    return dirs

def basis_from_direction(direction):
    forward = direction / np.linalg.norm(direction)
    up_ref = np.array([0.0, 1.0, 0.0])
    if abs(np.dot(forward, up_ref)) > 0.98:
        up_ref = np.array([1.0, 0.0, 0.0])
    right = np.cross(forward, up_ref); right = right / np.linalg.norm(right)
    up = np.cross(right, forward); up = up / np.linalg.norm(up)
    return right, up, forward

LOCAL_DIRS_32 = pixel_directions_32()

def world_direction(local_vec, direction):
    right, up, forward = basis_from_direction(direction)
    vec = local_vec[0]*right + local_vec[1]*forward + local_vec[2]*up
    return vec / np.linalg.norm(vec)


In [10]:
def raycast_surface(Y, eixo, geometry, pose):
    n = sensor["high_size"]
    depth = np.zeros((n, n), dtype=np.uint16)
    origin = pose["position"]
    max_range = sensor["max_range_m"]
    min_range = sensor["min_range_m"]
    steps = np.linspace(min_range, max_range, RAY_MARCH_STEPS)
    R = geometry["radius_m"]

    for j in range(n):
        for i in range(n):
            ray_dir = world_direction(LOCAL_DIRS_32[j, i], pose["direction"])
            hit_mm = 0
            prev_diff = None
            prev_t = None
            for t in steps:
                p = origin + ray_dir * t
                x, y, z = p
                r = np.sqrt(x**2 + z**2)
                if r > R:
                    break
                if x < eixo[0] or x > eixo[-1] or z < eixo[0] or z > eixo[-1]:
                    continue
                surface_y = bilinear_surface_y(Y, eixo, x, z)
                diff = y - surface_y
                if prev_diff is not None and prev_diff > 0 and diff <= 0:
                    hit_mm = int(round(prev_t * 1000.0))
                    break
                prev_diff = diff
                prev_t = t
            depth[j, i] = hit_mm
    return depth


In [ ]:
def downsample_32_to_8(depth32):
    n = sensor["low_size"]
    valido = (depth32 > 0).astype(np.float32)
    soma = cv2.resize(depth32.astype(np.float32) * valido, (n, n), interpolation=cv2.INTER_AREA)
    peso = cv2.resize(valido, (n, n), interpolation=cv2.INTER_AREA)
    media = np.divide(soma, peso, out=np.zeros_like(soma), where=peso > 0)
    return np.rint(media).astype(np.uint16)

def degrade_low_res(depth8, rng):
    result = depth8.astype(np.float32).copy()
    valid = result > 0
    noise = rng.normal(0.0, sensor["noise_sigma_mm"], size=result.shape)
    result[valid] += noise[valid]
    result[result < 0] = 0
    dropout_mask = rng.random(result.shape) < sensor["dropout_fraction"]
    result[dropout_mask] = 0
    return np.rint(result).astype(np.uint16)


In [12]:
for path in [SURFACES_YAML, SENSOR_YAML, SILOS_YAML]:
    shutil.copy2(path, CONFIG_DIR / path.name)

rng = np.random.default_rng(SEED)

for scene_index in range(DATASET_SIZE):
    scene_id = scene_index + 1
    silo = silos[scene_index % len(silos)]
    geometry = silo["geometry"]
    validar_silo(silo)

    eixo, X, Z, RAIO, mascara_silo = preparar_dominio(geometry["radius_m"], GRID_SIZE)
    surface_item = superficies[int(rng.integers(0, len(superficies)))]
    Y = gerar_superficie(surface_item, geometry["radius_m"], geometry["cylinder_height_m"], X, Z, RAIO, mascara_silo)

    pose = pose_aleatoria_valida(Y, eixo, geometry, rng)
    depth32 = raycast_surface(Y, eixo, geometry, pose)
    depth8 = degrade_low_res(downsample_32_to_8(depth32), rng)

    scene_dir = SCENES_DIR / f"scene_{scene_id:06d}"
    scene_dir.mkdir(parents=True, exist_ok=True)

    tifffile.imwrite(scene_dir / "depth_8x8.tif", depth8, compression=None)
    tifffile.imwrite(scene_dir / "depth_32x32.tif", depth32, compression=None)

    metadata = {
        "scene": {"id": scene_id, "seed": int(SEED + scene_index)},
        "silo": {
            "model": silo["model"],
            "source": silo["source"],
            "radius_m": geometry["radius_m"],
            "cylinder_height_m": geometry["cylinder_height_m"],
            "roof_height_m": geometry["roof_height_m"],
            "roof_top_radius_m": geometry["roof_top_radius_m"],
            "hopper_height_m": geometry["hopper_height_m"],
            "outlet_radius_m": geometry["outlet_radius_m"],
            "outlet_height_m": geometry["outlet_height_m"],
            "total_height_m": geometry["total_height_m"],
        },
        "surface": {"id": surface_item["id"], "type": surface_item["tipo"]},
        "sensor": {
            "model": sensor["model"],
            "config": {
                "fov_diagonal_deg": sensor["fov_diagonal_deg"],
                "min_range_m": sensor["min_range_m"],
                "max_range_m": sensor["max_range_m"],
                "noise_sigma_mm": sensor["noise_sigma_mm"],
                "dropout_fraction": sensor["dropout_fraction"],
            },
            "pose": {
                "sensor_height_fraction": pose["sensor_height_fraction"],
                "azimuth_deg": pose["azimuth_deg"],
                "radial_offset_fraction": pose["radial_offset_fraction"],
                "tilt_deg": pose["tilt_deg"],
            },
            "derived": {
                "x_m": float(pose["position"][0]),
                "y_m": float(pose["position"][1]),
                "z_m": float(pose["position"][2]),
                "surface_y_m": float(pose["surface_y_m"]),
                "roof_y_m": float(pose["roof_y_m"]),
                "roof_clearance_m": float(ROOF_CLEARANCE_M),
                "radial_offset_m": float(pose["radial_offset_m"]),
                "axis_x": float(pose["direction"][0]),
                "axis_y": float(pose["direction"][1]),
                "axis_z": float(pose["direction"][2]),
            },
        },
        "depth_encoding": {"dtype":"uint16","unit":"mm","invalid_value":0,"compression":"none"},
        "files": {"low_resolution":"depth_8x8.tif","high_resolution":"depth_32x32.tif"},
    }

    with (scene_dir / "metadata.yaml").open("w", encoding="utf-8") as f:
        yaml.safe_dump(metadata, f, sort_keys=False, allow_unicode=False)

print(f"Dataset gerado em: {DATASET_ROOT.resolve()}")
print(f"Cenas: {DATASET_SIZE}")


KeyboardInterrupt: 

In [13]:
for i in range(1, 8):
    scene_dir = SCENES_DIR / f"scene_{i:06d}"
    with (scene_dir / "metadata.yaml").open("r", encoding="utf-8") as f:
        meta = yaml.safe_load(f)
    print(f"scene_{i:06d}", "| silo =", meta["silo"]["model"], "| surface =", meta["surface"]["id"])


scene_000001 | silo = 36-02-60 | surface = 5
scene_000002 | silo = 30-04-60 | surface = 35
scene_000003 | silo = 30-4,5-60 | surface = 25
scene_000004 | silo = 36-03-60 | surface = 1
scene_000005 | silo = 36-3,5-60 | surface = 44
scene_000006 | silo = ovos_mombuca | surface = 26
scene_000007 | silo = 36-02-60 | surface = 42
